# Visualization of the Execution Process

In this chapter, we use the `visualize_computational_process.py` script to examine the execution process. This tool visualizes instruction scheduling and dependencies -- details that are not apparent from aggregated profile data alone. It accepts pipeline state JSON files for the **Dim2 single-layer** configuration as input; PBC is also supported, as its `machine_type` is `Dim2`.

In [ ]:
import json
import pathlib
import os
import platform
from collections import Counter

from IPython.display import Code

project_root = pathlib.Path("../../../..").resolve()
qret_path = project_root / "build" / "main"
if platform.system() == "Darwin": 
    gridsynth_path = project_root / "externals" / "bin" / "gridsynth_macos"
else:
    gridsynth_path = project_root / "externals" / "bin" / "gridsynth"

os.environ["GRIDSYNTH_PATH"] = str(gridsynth_path)
os.environ["PATH"] = str(qret_path) + os.pathsep + os.environ.get("PATH", "")

output_dir = pathlib.Path("../../tutorial-output")
output_dir.mkdir(exist_ok=True)

## 0. Preparing the Visualization Input
To prepare the visualization input files, we will first compile `Dim2` and `PBC` pipelines.

> [!IMPORTANT]
>
> `Dim3` / `DistributedDim2` / `DistributedDim3` are not supported by this visualization tool.


In [ ]:
dim2_pipeline_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_dim2_pipeline.yaml"
pbc_pipeline_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_pbc_pipeline.yaml"

!qret compile --verbose --pipeline {dim2_pipeline_path}
!qret compile --verbose --pipeline {pbc_pipeline_path}

## 1. A First Rough Comparison

In [ ]:
def count_types(path: str) -> Counter:
    data = json.loads(pathlib.Path(path).read_text())
    return Counter(inst["type"] for inst in data["program"])

for name, path in [("Dim2", output_dir / "tutorial_5_dim2.json"), ("PBC", output_dir / "tutorial_5_pbc.json")]:
    print(f"[{name}] {path}")
    for k, v in sorted(count_types(path).items()):
        print(f"  {k}: {v}")
    print()

## 2. Start The Visualizer
Launch the visualization UI using the command below, then load `tutorial_5_dim2.json` and `tutorial_5_pbc.json` one after the other to inspect them.

```sh
streamlit run ../../../../quration-visualizer/visualize_computational_process.py
```

Main Modes:
1. `Single`: Display the Timeline / Instructions / Dependency / Spatial 2D / Spatial 3D views individually.
2. `Unified`: Simultaneously display Timeline, Instructions, Dependency, Spatial 2D panels on a single dashboard.
3. `Playback`: Animates the execution in a 2D/3D view (Includes playback controls: `Start/Stop/Reset` and variable FPS).

Key Things to Observe:
1. Watch the instruction sequence execute chronologically inside the playback animation.
2. Adjust the current beat (logical clock cycle) index in the sidebar panel to see how the system progresses through the instruction dependency graph.
3. Compare the physical hardware footprint to observe the distinct differences in chip occupancy between standard Dim2 and PBC mode.

## 3. Key PBC Observation Points
When analyzing the Pauli-Based Computation (PBC) pipeline, keep the following execution characteristics in mind:
- **End-of-Circuit Measurement Alignment**: PBC generates the instruction sequence under the assumption that all qubits are measured at the very end of the circuit.
- **Key Visual Differences**: You can easily see how this differs from the standard mode by focusing on the heavy concentration of measurement instructions in the final stages of the timeline, as well as the distribution of classical dependencies (such as `XOR` operations).
